**고객 문의 이메일 3건에 대한 답장을 한 번에 생성하고, 호출별 토큰 사용량과 총비용을 출력하라.**

* ChatPromptTemplate.from_messages로 system · human 메시지를 구성한다.

    * system: 상담원 역할, 회사명, 서명

    * human: 고객 이름, 문의 내용

* 회사명(SOSIN Shop)과 서명은 partial로 미리 채워, 호출할 때는 고객 이름과 문의 내용만 넘기게 한다. input_variables로 남은 변수를 확인한다.

* 고객이 쓴 언어와 같은 언어로 답장하도록 지시한다. 테스트 데이터에 영어 문의 1건을 포함한다.

* 체인은 prompt | llm으로 구성하고 batch로 3건을 동시에 처리한다.

* 각 응답의 usage_metadata로 입력 · 출력 토큰을 누적하고, gpt-6-luna 단가(입력 $0.10, 출력 $0.50 / 1M)로 총비용을 계산한다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
llm = ChatOpenAI(model="gpt-6-luna")

In [2]:
# system: 상담원 역할 + 회사명 + 서명, human: 고객 이름 + 문의 내용
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 {company}의 고객 상담원임. 고객 문의에 정중하고 간결한 답장 이메일을 작성함. "
     "반드시 고객이 쓴 언어와 같은 언어로 답장할 것. "
     "답장 마지막에는 아래 서명을 그대로 붙일 것.\n{signature}"),
    ("human", "고객 이름: {customer_name}\n문의 내용: {inquiry}"),
])

# 회사명과 서명은 partial로 미리 채움
prompt = prompt.partial(company="SOSIN Shop", signature="SOSIN Shop 고객지원팀 드림")
print("남은 입력 변수:", prompt.input_variables)

남은 입력 변수: ['customer_name', 'inquiry']


In [3]:
# 테스트 데이터 (영어 문의 1건 포함) 및 batch 처리
inquiries = [
    {"customer_name": "김민수",
     "inquiry": "지난주에 주문한 운동화가 아직 도착하지 않았어요. 배송 상태를 확인할 수 있을까요?"},
    {"customer_name": "John Smith",
     "inquiry": "I received a jacket in the wrong size. How can I exchange it for a medium?"},
    {"customer_name": "이서연",
     "inquiry": "주문한 상품 중 하나만 부분 취소하고 싶은데 가능한가요?"},
]

chain = prompt | llm
responses = chain.batch(inquiries)

In [4]:
# 토큰 누적 및 총비용 계산
# gpt-6-luna 단가 (1M 토큰 기준, USD)
PRICE_IN = 0.10
PRICE_OUT = 0.50

total_in = total_out = 0
for i, (item, res) in enumerate(zip(inquiries, responses), start=1):
    usage = res.usage_metadata
    total_in += usage["input_tokens"]
    total_out += usage["output_tokens"]
    print(f"[{i}] {item['customer_name']}")
    print(res.content)
    print(f"  -> 입력 {usage['input_tokens']} / 출력 {usage['output_tokens']} 토큰\n")

total_cost = total_in / 1_000_000 * PRICE_IN + total_out / 1_000_000 * PRICE_OUT
print(f"총 입력 토큰: {total_in}")
print(f"총 출력 토큰: {total_out}")
print(f"총비용: ${total_cost:.6f}")

[1] 김민수
김민수 고객님, 안녕하세요.

지난주 주문하신 운동화가 아직 도착하지 않아 불편을 드려 죄송합니다. 배송 상태를 확인할 수 있도록 주문번호를 알려주시면 확인 후 안내해 드리겠습니다.

SOSIN Shop 고객지원팀 드림
  -> 입력 110 / 출력 97 토큰

[2] John Smith
Hi John,

We’re sorry you received the wrong size. Please reply with your order number, and we’ll help arrange an exchange for a medium.

Best regards,  
SOSIN Shop 고객지원팀 드림
  -> 입력 103 / 출력 139 토큰

[3] 이서연
이서연 고객님, 안녕하세요.

주문 상품 중 일부만 취소하실 수 있습니다. 주문번호와 취소를 원하시는 상품명을 알려주시면 취소 가능 여부를 확인해 안내해 드리겠습니다. 상품이 이미 출고된 경우 취소가 어려울 수 있는 점 양해 부탁드립니다.

SOSIN Shop 고객지원팀 드림
  -> 입력 102 / 출력 144 토큰

총 입력 토큰: 315
총 출력 토큰: 380
총비용: $0.000222


---

**자유롭게 쓴 요리 설명 3개에서 레시피 정보를 구조화하여 추출하고, 두 가지 구조화 방법을 비교하라.**

* Pydantic 스키마

    * Ingredient(name, amount)

    * Recipe(dish, servings: int, minutes: int, ingredients: list[Ingredient], steps: list[str])

    * 필드마다 Field(description=...)으로 설명을 붙인다.

* 방법 1: PydanticOutputParser — get_format_instructions()를 partial로 system 메시지에 넣고 prompt | llm | parser로 구성한다.

* 방법 2: llm.with_structured_output(Recipe)로 구성한다.

* 두 방법 모두 파싱 실패(OutputParserException)를 처리하고, 방법별 성공 건수와 추출 결과(인분 · 시간 · 재료 수 · 단계 수)를 출력한다.

* 테스트 데이터는 분량이 명시되지 않은 재료(예: "설탕을 뿌립니다")를 포함한 다음의 데이터를 사용한다.
```
texts = [
    "김치볶음밥 2인분이에요. 밥 두 공기, 김치 한 컵, 스팸 반 캔, 계란 2개 준비하세요. "
    "팬에 스팸과 김치를 볶다가 밥을 넣고 5분 더 볶고, 계란 프라이를 올리면 끝. 15분이면 충분해요.",
    "된장찌개(3인분, 약 25분): 물 600ml에 된장 2큰술을 풀고 감자, 호박, 두부를 넣어 끓입니다. 마지막에 대파를 넣어요.",
    "토스트 1인분. 식빵 2장에 버터를 바르고 구운 뒤 설탕을 뿌립니다. 5분 걸려요.",
]
```

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
llm = ChatOpenAI(model="gpt-6-luna")

In [5]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException


# 스키마 정의
class Ingredient(BaseModel):
    name: str = Field(description="재료 이름")
    amount: str = Field(description="재료 분량. 원문에 분량이 없으면 '미정'으로 적음")


class Recipe(BaseModel):
    dish: str = Field(description="요리 이름")
    servings: int = Field(description="인분 수")
    minutes: int = Field(description="총 조리 시간(분)")
    ingredients: list[Ingredient] = Field(description="재료 목록")
    steps: list[str] = Field(description="조리 순서. 한 단계당 한 문장")

In [6]:
# 테스트 데이터
texts = [
    "김치볶음밥 2인분이에요. 밥 두 공기, 김치 한 컵, 스팸 반 캔, 계란 2개 준비하세요. "
    "팬에 스팸과 김치를 볶다가 밥을 넣고 5분 더 볶고, 계란 프라이를 올리면 끝. 15분이면 충분해요.",
    "된장찌개(3인분, 약 25분): 물 600ml에 된장 2큰술을 풀고 감자, 호박, 두부를 넣어 끓입니다. 마지막에 대파를 넣어요.",
    "토스트 1인분. 식빵 2장에 버터를 바르고 구운 뒤 설탕을 뿌립니다. 5분 걸려요.",
]

In [7]:
# 방법 1: PydanticOutputParser (format_instructions를 partial로 주입)
parser = PydanticOutputParser(pydantic_object=Recipe)
prompt_parser = ChatPromptTemplate.from_messages([
    ("system", "요리 설명에서 레시피 정보를 추출함.\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())
chain_parser = prompt_parser | llm | parser

In [8]:
# 방법 2: with_structured_output
prompt_struct = ChatPromptTemplate.from_messages([
    ("system", "요리 설명에서 레시피 정보를 추출함. 분량이 명시되지 않은 재료의 amount는 '미정'으로 적음."),
    ("human", "{text}"),
])
chain_struct = prompt_struct | llm.with_structured_output(Recipe)

In [9]:
def extract_all(chain, texts):
    """건별로 호출하여 파싱 실패 시 None으로 기록"""
    results = []
    for i, t in enumerate(texts, start=1):
        try:
            results.append(chain.invoke({"text": t}))
        except OutputParserException as e:
            print(f"  [{i}] 파싱 실패: {str(e)[:100]}")
            results.append(None)
    return results


def report(title, results):
    ok = [r for r in results if r is not None]
    print(f"=== {title}: 성공 {len(ok)}/{len(results)} ===")
    for i, r in enumerate(results, start=1):
        if r is None:
            print(f"  [{i}] 실패")
            continue
        print(f"  [{i}] {r.dish} | {r.servings}인분 | {r.minutes}분 | "
              f"재료 {len(r.ingredients)}개 | 단계 {len(r.steps)}개")
    print()

In [10]:
# 두 방법 실행 및 결과 비교
results_parser = extract_all(chain_parser, texts)
results_struct = extract_all(chain_struct, texts)

report("방법 1 PydanticOutputParser", results_parser)
report("방법 2 with_structured_output", results_struct)

=== 방법 1 PydanticOutputParser: 성공 3/3 ===
  [1] 김치볶음밥 | 2인분 | 15분 | 재료 4개 | 단계 3개
  [2] 된장찌개 | 3인분 | 25분 | 재료 6개 | 단계 3개
  [3] 토스트 | 1인분 | 5분 | 재료 3개 | 단계 2개

=== 방법 2 with_structured_output: 성공 3/3 ===
  [1] 김치볶음밥 | 2인분 | 15분 | 재료 4개 | 단계 3개
  [2] 된장찌개 | 3인분 | 25분 | 재료 6개 | 단계 3개
  [3] 토스트 | 1인분 | 5분 | 재료 3개 | 단계 1개



---

**사내 메신저로 들어오는 구어체 업무 요청을 표준 티켓 JSON으로 변환하는 체인을 만들어라.**
```
examples = [
    {"request": "로그인 페이지가 자꾸 흰 화면만 떠요 급해요",
     "ticket": '{"team": "개발", "title": "[개발] 로그인 페이지 빈 화면 오류 수정", "priority": "high"}'},
    {"request": "다음 달 신입 온보딩 자료 좀 업데이트 해주실 수 있나요",
     "ticket": '{"team": "인사", "title": "[인사] 신입 온보딩 자료 업데이트", "priority": "low"}'},
    {"request": "3분기 광고비 정산 내역이 안 맞는 것 같아요 이번 주 안에 확인 부탁",
     "ticket": '{"team": "재무", "title": "[재무] 3분기 광고비 정산 내역 확인", "priority": "medium"}'},
]

requests = [
    "회의실 프로젝터가 안 켜져요 오후 2시 회의 전에 봐주세요",
    "연말정산 서류 제출 기한이 언제까지예요?",
    "결제 API 응답이 10초 넘게 걸려서 고객 불만이 들어오고 있어요",
    "사무실 커피머신 원두 주문 부탁드립니다",
    "법인카드 한도 상향 신청하고 싶어요",
]
```
* 출력 형식: {"team": 개발/인사/재무/총무 중 하나, "title": "[팀] 명사형 작업명", "priority": low/medium/high}

* FewShotChatMessagePromptTemplate으로 요청 → 티켓 예시 3개를 구성한다. 예시의 팀과 우선순위는 서로 다르게 구성한다.

* system 메시지 + few-shot 예시 + human 입력으로 ChatPromptTemplate을 만들고, JsonOutputParser로 dict를 받는다.

* 업무 요청 5개를 batch로 처리하고, 팀 · 우선순위 허용값과 title 형식([로 시작)을 코드로 검사해 O/X로 출력한다.

* 같은 프롬프트를 StrOutputParser와 연결한 체인으로 요청 1건을 stream 출력한다.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
llm = ChatOpenAI(model="gpt-6-luna")

In [11]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

examples = [
    {"request": "로그인 페이지가 자꾸 흰 화면만 떠요 급해요",
     "ticket": '{"team": "개발", "title": "[개발] 로그인 페이지 빈 화면 오류 수정", "priority": "high"}'},
    {"request": "다음 달 신입 온보딩 자료 좀 업데이트 해주실 수 있나요",
     "ticket": '{"team": "인사", "title": "[인사] 신입 온보딩 자료 업데이트", "priority": "low"}'},
    {"request": "3분기 광고비 정산 내역이 안 맞는 것 같아요 이번 주 안에 확인 부탁",
     "ticket": '{"team": "재무", "title": "[재무] 3분기 광고비 정산 내역 확인", "priority": "medium"}'},
]

requests = [
    "회의실 프로젝터가 안 켜져요 오후 2시 회의 전에 봐주세요",
    "연말정산 서류 제출 기한이 언제까지예요?",
    "결제 API 응답이 10초 넘게 걸려서 고객 불만이 들어오고 있어요",
    "사무실 커피머신 원두 주문 부탁드립니다",
    "법인카드 한도 상향 신청하고 싶어요",
]

In [12]:
# few-shot 예시: 요청(human) -> 티켓(ai)
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{request}"),
    ("ai", "{ticket}"),
])
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# JSON 예시의 중괄호는 {{ }}로 이스케이프
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "사내 메신저의 구어체 업무 요청을 표준 티켓 JSON으로 변환함. JSON 외의 텍스트는 출력하지 않음.\n"
     '출력 형식: {{"team": "개발|인사|재무|총무 중 하나", '
     '"title": "[팀] 명사형 작업명", "priority": "low|medium|high"}}'),
    few_shot,
    ("human", "{request}"),
])

In [13]:
# JsonOutputParser 체인 및 batch 처리
chain = prompt | llm | JsonOutputParser()
tickets = chain.batch([{"request": r} for r in requests])

In [14]:
# 팀 · 우선순위 허용값과 title 형식 검사
TEAMS = {"개발", "인사", "재무", "총무"}
PRIORITIES = {"low", "medium", "high"}

for req, t in zip(requests, tickets):
    team_ok = t.get("team") in TEAMS
    prio_ok = t.get("priority") in PRIORITIES
    title_ok = isinstance(t.get("title"), str) and t["title"].startswith("[")
    print(f"요청: {req}")
    print(f"  티켓: {t}")
    print(f"  team {'O' if team_ok else 'X'} | "
          f"priority {'O' if prio_ok else 'X'} | "
          f"title {'O' if title_ok else 'X'}")

요청: 회의실 프로젝터가 안 켜져요 오후 2시 회의 전에 봐주세요
  티켓: {'team': '총무', 'title': '[총무] 회의실 프로젝터 점검', 'priority': 'high'}
  team O | priority O | title O
요청: 연말정산 서류 제출 기한이 언제까지예요?
  티켓: {'team': '인사', 'title': '[인사] 연말정산 서류 제출 기한 확인', 'priority': 'low'}
  team O | priority O | title O
요청: 결제 API 응답이 10초 넘게 걸려서 고객 불만이 들어오고 있어요
  티켓: {'team': '개발', 'title': '[개발] 결제 API 응답 지연 개선', 'priority': 'high'}
  team O | priority O | title O
요청: 사무실 커피머신 원두 주문 부탁드립니다
  티켓: {'team': '총무', 'title': '[총무] 커피머신 원두 주문', 'priority': 'low'}
  team O | priority O | title O
요청: 법인카드 한도 상향 신청하고 싶어요
  티켓: {'team': '재무', 'title': '[재무] 법인카드 한도 상향 신청', 'priority': 'medium'}
  team O | priority O | title O


In [15]:
# 같은 프롬프트 + StrOutputParser, 요청 1건 stream 출력
str_chain = prompt | llm | StrOutputParser()
for chunk in str_chain.stream({"request": requests[0]}):
    print(chunk, end="", flush=True)
print()

{"team": "총무", "title": "[총무] 회의실 프로젝터 작동 오류 점검", "priority": "high"}
